In [1]:
%pip install requests mysql-connector-python

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import time
import shutil
import requests

## Часть 2. Веб-приложение в Docker Compose

стурктура проекта будет такой. дальше я ее поэтапоно создаю

```text
app_chinook/
├── Dockerfile
├── docker-compose.yml
├── main.py
├── requirements.txt
└── init/
    ├── 01_chinook.sql
    └── 02_customer_feedback.sql
```

In [7]:
# отделтная папка проекта для приложения
os.makedirs("app_chinook/init", exist_ok=True)


for name in ["init/01_chinook.sql", "chinook.sql"]:
    if os.path.exists(name):
        shutil.copy(name, "app_chinook/init/01_chinook.sql")
        print("файл скопирован из", name)
        break


файл скопирован из chinook.sql


### Таблица для отзывов

In [8]:
%%writefile app_chinook/init/02_customer_feedback.sql
-- таблица для отзывов, нужна эндпоинту POST /api/feedback
-- этот файл выполнится сразу после 01_chinook.sql (файлы идут по алфавиту)
USE Chinook_AutoIncrement;

CREATE TABLE IF NOT EXISTS customer_feedback (
    feedback_id INT AUTO_INCREMENT PRIMARY KEY,
    customer_id INT NOT NULL,
    track_id INT NOT NULL,
    rating INT NOT NULL CHECK (rating BETWEEN 1 AND 5),
    comment TEXT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (customer_id) REFERENCES Customer(CustomerId),
    FOREIGN KEY (track_id) REFERENCES Track(TrackId)
);

Writing app_chinook/init/02_customer_feedback.sql


### Само приложение на FastAPI. 


Каждая функция с `@app.get(...)` или `@app.post(...)` отвечает за один адрес из задания

In [ ]:
%%writefile app_chinook/main.py
# подключаем нужные библиотеки
import os

import mysql.connector  # чтобы работать с базой mysql
from fastapi import FastAPI  # сам fastapi
from pydantic import BaseModel  # нужен, чтобы описать тело post запроса

# создаем приложение, все адреса ниже привязываются к нему
app = FastAPI()

# настройки подключения к базе
db_host = os.getenv("DB_HOST", "localhost")
db_port = int(os.getenv("DB_PORT", 3306))
db_user = os.getenv("DB_USER", "root")
db_password = os.getenv("DB_PASSWORD", "root")
db_name = os.getenv("DB_NAME", "Chinook_AutoIncrement")


def get_connection():
    # чтобы не писать одно и то же в каждой функции,
    # подключение вынесла в отдельную функцию
    # каждый раз открываем новое подключение к базе
    return mysql.connector.connect(
        host=db_host, port=db_port, user=db_user,
        password=db_password, database=db_name
    )


# типичный отзыв

class Feedback(BaseModel):
    customer_id: int
    track_id: int
    rating: int
    comment: str = None


# 1. поиск треков по жанру
@app.get("/api/tracks")
def get_tracks(genre: str):
    # genre берется из адреса: /api/tracks?genre=Rock
    con = get_connection()
    # dictionary=True, чтобы строки приходили как словари (название столбца: значение)
    cur = con.cursor(dictionary=True)
    # %s - место для значения, жанр передаем отдельно вторым аргументом
    cur.execute("""
        SELECT t.TrackId, t.Name, t.UnitPrice
        FROM Track AS t
        JOIN Genre g ON g.GenreId = t.GenreId
        WHERE g.Name = %s
    """, (genre,))


    tracks = cur.fetchall()
    con.close()
    return tracks


# 2. альбомы артиста
@app.get("/api/artists/{artist_id}/albums")
def get_albums(artist_id: int):
    # artist_id берется из самого адреса: /api/artists/1/albums
    con = get_connection()
    cur = con.cursor(dictionary=True)
    cur.execute("SELECT AlbumId, Title FROM Album WHERE ArtistId = %s", (artist_id,))
    albums = cur.fetchall()
    con.close()
    # если у артиста нет альбомов, вернется пустой список
    return albums


# 3. отчет по клиенту
@app.get("/api/clients/{client_id}/report")
def get_report(client_id: int):
    con = get_connection()
    cur = con.cursor(dictionary=True)

    # первый запрос: имя, email, количество счетов и общая сумма покупок
    cur.execute("""
        SELECT c.FirstName, c.LastName, c.Email,
               COUNT(i.InvoiceId) AS invoice_count,
               SUM(i.Total) AS total_sum
        FROM Customer AS c
        JOIN Invoice i ON i.CustomerId = c.CustomerId
        WHERE c.CustomerId = %s
        GROUP BY c.CustomerId
    """, (client_id,))
    # fetchone берет одну строку (клиент один)
    report = cur.fetchone()

    # если такого клиента нет, fetchone вернет None
    if report is None:
        con.close()
        return {"error": "клиент не найден"}

    # второй запрос: жанр, треков которого клиент купил больше всего
    # группируем покупки по жанрам, сортируем по убыванию и берем первую строку
    cur.execute("""
        SELECT g.Name AS genre, SUM(il.Quantity) AS bought
        FROM Invoice i
        JOIN InvoiceLine AS il ON il.InvoiceId = i.InvoiceId
        JOIN Track AS t ON t.TrackId = il.TrackId
        JOIN Genre AS g ON g.GenreId = t.GenreId
        WHERE i.CustomerId = %s
        GROUP BY g.Name
        ORDER BY bought DESC
        LIMIT 1
    """, (client_id,))
    top = cur.fetchone()
    # добавляем любимый жанр к результату первого запроса
    report["top_genre"] = top["genre"]

    con.close()
    return report


# 4. добавление отзыва
@app.post("/api/feedback")
def add_feedback(feedback: Feedback):
    # данные отзыва приходят в теле запроса (json), fastapi кладет их в feedback
    # рейтинг от 1 до 5 проверяет сама база (CHECK в таблице)
    con = get_connection()
    cur = con.cursor()
    cur.execute(
        "INSERT INTO customer_feedback (customer_id, track_id, rating, comment) "
        "VALUES (%s, %s, %s, %s)",
        (feedback.customer_id, feedback.track_id, feedback.rating, feedback.comment)
    )
    # commit нужен, чтобы запись сохранилась в базе
    con.commit()
    # lastrowid - id только что добавленной записи
    new_id = cur.lastrowid
    con.close()
    return {"feedback_id": new_id, "status": "отзыв добавлен"}


Overwriting app_chinook/main.py


### Зависимости и Dockerfile для сервиса `web`.

`docker-compose.yml`: два сервиса. `db` это MySQL, `web` это наше приложение. 

In [18]:
%%writefile app_chinook/docker-compose.yml
services:

  # контейнер с базой
  db:
    image: mysql:8.0
    environment:
      MYSQL_ROOT_PASSWORD: root
    ports:
      - "3308:3306"
    volumes:
      - mysql_data:/var/lib/mysql
      # sql файлы из этой папки выполнятся при первом запуске
      - ./init:/docker-entrypoint-initdb.d

  # контейнер с нашим приложением
  web:
    build: .
    ports:
      - "8000:8000"
    environment:
      # db - имя сервиса с базой, внутри сети docker по нему можно подключиться
      DB_HOST: db
      DB_PORT: 3306
      DB_USER: root
      DB_PASSWORD: root
      DB_NAME: Chinook_AutoIncrement
    depends_on:
      - db

volumes:
  mysql_data:

Writing app_chinook/docker-compose.yml


In [23]:
%%writefile app_chinook/requirements.txt
fastapi
uvicorn
mysql-connector-python

Writing app_chinook/requirements.txt


In [24]:
%%writefile app_chinook/Dockerfile
# берем образ с питоном
FROM python:3.12-slim

WORKDIR /app

# ставим библиотеки
COPY requirements.txt .
RUN pip install -r requirements.txt

# копируем наше приложение
COPY main.py .

# запускаем сервер, 0.0.0.0 чтобы было видно снаружи контейнера
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]

Writing app_chinook/Dockerfile


### Запуск

In [25]:
!ls -la app_chinook

total 40
drwxr-xr-x@ 7 veramonina  staff   224 Oct 20 23:46 .
drwxr-xr-x@ 8 veramonina  staff   256 Oct 20 23:35 ..
-rw-r--r--@ 1 veramonina  staff   418 Oct 20 23:46 Dockerfile
-rw-r--r--@ 1 veramonina  staff   777 Oct 20 23:42 docker-compose.yml
drwxr-xr-x@ 4 veramonina  staff   128 Oct 20 23:38 init
-rw-r--r--@ 1 veramonina  staff  6411 Oct 20 23:42 main.py
-rw-r--r--@ 1 veramonina  staff    39 Oct 20 23:46 requirements.txt


In [26]:
!docker compose -f app_chinook/docker-compose.yml up -d --build


[+] Building 0.0s (0/1)                                                         
 => [internal] load local bake definitions                                 0.0s
[+] Building 0.2s (1/1)                                                         
 => [internal] load local bake definitions                                 0.0s
 => => reading from stdin 544B                                             0.0s
[+] Building 0.3s (1/1)                                                         
 => [internal] load local bake definitions                                 0.0s
 => => reading from stdin 544B                                             0.0s
[+] Building 0.5s (1/1)                                                         
 => [internal] load local bake definitions                                 0.0s
 => => reading from stdin 544B                                             0.0s
[+] Building 0.6s (1/1)                                                         
 => [internal] load local bake def

In [27]:
!docker compose -f app_chinook/docker-compose.yml ps

NAME                IMAGE             COMMAND                  SERVICE   CREATED         STATUS         PORTS
app_chinook-db-1    mysql:8.0         "docker-entrypoint.s…"   db        7 minutes ago   Up 7 minutes   0.0.0.0:3308->3306/tcp, [::]:3308->3306/tcp
app_chinook-web-1   app_chinook-web   "uvicorn main:app --…"   web       7 minutes ago   Up 7 minutes   0.0.0.0:8000->8000/tcp, [::]:8000->8000/tcp


In [28]:
import requests

base = "http://localhost:8000"

for i in range(60):
    try:
        r = requests.get(base + "/api/tracks", params={"genre": "Rock"})
        if r.status_code == 200:
            print("Приложение и база готовы")
            break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(5)

Приложение и база готовы


In [29]:
# 1. поиск треков по жанру
r = requests.get(base + "/api/tracks", params={"genre": "Rock"})
print(r.status_code)
print("треков:", len(r.json()))
print(r.json()[:3])

200
треков: 1297
[{'TrackId': 1, 'Name': 'For Those About To Rock (We Salute You)', 'UnitPrice': 0.99}, {'TrackId': 2, 'Name': 'Balls to the Wall', 'UnitPrice': 0.99}, {'TrackId': 3, 'Name': 'Fast As a Shark', 'UnitPrice': 0.99}]


In [30]:
# 2. альбомы артиста с id 1
r = requests.get(base + "/api/artists/1/albums")
print(r.status_code)
print(r.json())

200
[{'AlbumId': 1, 'Title': 'For Those About To Rock We Salute You'}, {'AlbumId': 4, 'Title': 'Let There Be Rock'}]


In [31]:
# 3. отчет по клиенту с id 1
r = requests.get(base + "/api/clients/1/report")
print(r.status_code)
print(r.json())

200
{'FirstName': 'Luís', 'LastName': 'Gonçalves', 'Email': 'luisg@embraer.com.br', 'invoice_count': 7, 'total_sum': 39.62, 'top_genre': 'Rock'}


In [32]:
# 4. добавляем отзыв
r = requests.post(base + "/api/feedback",
                  json={"customer_id": 1, "track_id": 1, "rating": 5, "comment": "Отлично"})
print(r.status_code)
print(r.json())

200
{'feedback_id': 1, 'status': 'отзыв добавлен'}


Документация API, которую FastAPI строит сам: http://localhost:8000/docs

Остановить приложение (volume с данными при этом не удаляется):

In [34]:
!docker compose -f app_chinook/docker-compose.yml down

[+] down 0/1
 ⠋ Container app_chinook-web-1 Stopping                                     0.0s

 ⠙ Container app_chinook-web-1 Stopping                                     0.1s

 ⠹ Container app_chinook-web-1 Stopping                                     0.2s

 ⠸ Container app_chinook-web-1 Stopping                                     0.3s

 ⠼ Container app_chinook-web-1 Stopping                                     0.4s

 ⠴ Container app_chinook-web-1 Stopping                                     0.5s

 ⠦ Container app_chinook-web-1 Stopping                                     0.6s

 ⠧ Container app_chinook-web-1 Removing                                     0.7s
[+] down 1/2
 ✔ Container app_chinook-web-1 Removed                                      0.7s
 ⠋ Container app_chinook-db-1  Stopping                                     0.1s


 ⠙ Container app_chinook-db-1  Stopping                                     0.2s


 ⠹ Container app_chinook-db-1  Stopping                                 

##  Ошибки в коде приложения (Заданеи3)


**1. SQL-запросы**
- Что не так: значения подставляются в запрос напрямую, через конкатенацию (`"... = " + str(customer_id)`) и f-строки (`{amount}`, `{from_id}`, `{limit}`).
- Почему это проблема: данные пользователя могут стать частью SQL-кода. Мне кажется, что если в `/api/transfer` преедать, например `from_customer_id = "1 OR 1=1"`, то условие `WHERE` станет истинным для всех строк, и сумма спишется у всех клиентов, чего мы бы не хотели. 
- Как исправить: использовать параметризованные запросы: `cur.execute("... WHERE CustomerId = %s", (customer_id,))`, как мы делали в 1 части дз

**2. Соединения с базой не закрываются**
- Что не так: в `get_customer` и `get_tracks` нет `con.close()`, а в `transfer` закрытие пропускается при любой ошибке.
- Почему это проблема: каждый запрос оставляет открытое соединени, поэтому при большой нагрузке новые подключения перестанут обрабатываться и приниматьсся базой данныъ.
- Как исправить: закрывать соединение в блоке `finally`.

**3. Параметры подключения записаны в коде**
- Что не так: `host="localhost", user="root", password="", database="chinook"` указаны прямо в функциях.
- Почему это проблема: пользователь root с пустым паролем даёт полный доступ к базе любому человеку + пароль в коде может попасть в итоговый репозиторий + в контейнере `localhost` означает сам контейнер приложения, а не контейнер с MySQL   + имя базы `chinook` не совпадает с именем из дампа (`Chinook_AutoIncrement`).
- Как исправить: брать настройки из переменных окружения, в Docker Compose указывать имя сервиса вместо `localhost`

**4. Дублирование кода подключения**
- Что не так: одинаковый блок `mysql.connector.connect(...)` скопирован в три функции, и новое соединение открывается при каждом запросе.
- Почему это проблема: при изменении настроек надо не забыть исправить код в 3 местах, а не в 1, что может привести к ошибкам. Также мне кажется, что если на сайт будет высокая нагрузка, то она увеличися в 3 раза и наше приложение будет виснуть. 
- Как исправить: вынести подключение в одну функцию `get_connection()`.

**5. Использование `SELECT *`**
- Что не так: в `get_customer` и `get_tracks` из таблицы выбираются все столбцы подряд, а jsonify отдаёт строки в виде обычных списков, без имён полей.
- Почему это проблема: в ответ API попадают лишние персональные данные клиента (адрес, телефон, email)
- Как исправить: перечислить только нужные столбцы

**6. Несуществующий клиент возвращает код 200**
- Что не так: если клиента с таким id нет, `fetchall()` возвращает пустой список, а сервер отвечает кодом 200.
- Почему это проблема: по ответу невозможно отличить 2 ошибки (ошибки запроса и ошибки отсуствия клиента в базе данных).
- Как исправить: использовать `fetchone()` и при результате `None` возвращать ошибку с кодом 404, как это обычно делается.

**7. Перевод денег выполняется без обработки ошибок и отката операции**
- Что не так: два `UPDATE` выполняются подряд, а `try/except` и `rollback()` в коде нет, как мы делали в 1 части дз
- Почему это проблема: если второй запрос завершится ошибкой, первый останется незавершённым, и деньги будут списаны, но не зачислены. Соединение при этом останется открытым.
- Как исправить:сделать try-except, при ошибке вызывать `con.rollback()`, а закрытие соединения вынести в `finally`.

**8. В таблице `Customer` нет столбца `Balance`**
- Что не так: запросы в `/api/transfer` обращаются к столбцу `Balance`, которого в базе Chinook нет.
- Почему это проблема: оба запроса завершатся ошибкой `Unknown column 'Balance'`
- Как исправить: добавить в таблицу столбец, например `Balance DECIMAL(10,2) NOT NULL DEFAULT 0`


